In [1]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv(
    "./data/safety_notice_processed.csv",
    encoding="utf-8-sig"
)

# 2. 작성일을 날짜형으로 변환
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
)

# 3. 연도 / 월 만들기
df["연도"] = df["안전공지_작성일"].dt.year.astype("Int64")
df["월"] = df["안전공지_작성일"].dt.month.astype("Int64")

# 4. 연도 + 월 + 대륙별 공지건수 집계
monthly_continent = (
    df.dropna(subset=["대륙명", "연도", "월"])
      .groupby(["연도", "월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
)

display(monthly_continent.head(10))

,연도,월,대륙명,공지건수
0,2025,3,미주,5
1,2025,3,아주,7
2,2025,3,아프리카,20
3,2025,3,유럽,22
4,2025,3,중동,3
5,2025,4,미주,9
6,2025,4,아주,19
7,2025,4,아프리카,13
8,2025,4,유럽,28
9,2025,4,중동,2


In [ ]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# 1. 선택 위젯
# -------------------------

year_dropdown = widgets.Dropdown(
    options=["전체", 2025, 2026],
    value="전체",
    description="연도:"
)

continent_dropdown = widgets.Dropdown(
    options=["전체"] + sorted(
        monthly_continent["대륙명"].dropna().unique().tolist()
    ),
    value="전체",
    description="대륙:"
)


# -------------------------
# 2. 그래프 + 상세 데이터
# -------------------------

def draw_dashboard(year, continent):

    chart_df = monthly_continent.copy()

    # 연도 필터
    if year != "전체":
        chart_df = chart_df[
            chart_df["연도"] == year
        ]

    # 대륙 필터
    if continent != "전체":
        chart_df = chart_df[
            chart_df["대륙명"] == continent
        ]

    # 연월 표시
    chart_df["연월표시"] = chart_df.apply(
        lambda x: f"{str(x['연도'])[2:]}년 {x['월']}월",
        axis=1
    )

    # -------------------------
    # 그래프
    # -------------------------

    fig = px.line(
        chart_df,
        x="연월표시",
        y="공지건수",
        color="대륙명",
        markers=True,
        title="월별 대륙별 안전공지 추이"
    )

    fig.update_layout(
        xaxis_title="연월",
        yaxis_title="안전공지 건수",
        legend_title="대륙",
        hovermode="x unified"
    )

    fig.show()

    # -------------------------
    # 상세 데이터
    # -------------------------

    detail_df = df.copy()

    if year != "전체":
        detail_df = detail_df[
            detail_df["연도"] == year
        ]

    if continent != "전체":
        detail_df = detail_df[
            detail_df["대륙명"] == continent
        ]

# -------------------------
# 3. 대시보드 실행
# -------------------------

dashboard = widgets.interactive(
    draw_dashboard,
    year=year_dropdown,
    continent=continent_dropdown
)

display(dashboard)

interactive(children=(Dropdown(description='연도:', options=('전체', 2025, 2026), value='전체'), Dropdown(descriptio…

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# 임베딩 모델
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
)


# 기존 Vector DB 불러오기
vector_db = Chroma(
    persist_directory="data/vector_db",
    embedding_function=embeddings,
    collection_name="travel_safety",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
data = vector_db.get()

for metadata in data["metadatas"]:
    country = metadata.get("국가명", "")
    
    if "마카오" in country:
        print(metadata)

In [9]:
data = vector_db.get(
    where={"국가명": "홍콩"}
)

print("홍콩 청크 개수:", len(data["ids"]))

for i in range(len(data["ids"])):
    print("\nID:", data["ids"][i])
    print("metadata:", data["metadatas"][i])
    print("내용:", data["documents"][i][:500])

홍콩 청크 개수: 51

ID: i_00492
metadata: {'영문국가명': 'Hongkong', 'source': 'incident', 'ISO코드': 'HK', '국가명': '홍콩', '작성일': '2024-02-08', 'chunk_id': 'i_00492', '대륙명': '아주'}
내용: 사건ㆍ사고의 유형
ㅇ 도난 및 분실사건이 많이 발생하고 있습니다.
ㅇ 공항버스 이용 후 하차시 여행가방을 두고 내리는 경우가 많습니다.
ㅇ 침 뱉기, 쓰레기 투기, 흡연, 대중교통 내 음식물 섭취 등 기초질서 위반에 대해서 고액의 벌금이 부과됩니다.
ㅇ 교통사고의 경우, 법원의 판결 전까지는 피해자가 병원비 등 일체의 비용을 부담해야하며 재판결과에 따라 피해보상을 별도로 진행해야합니다. 가능하면 반드시 여행자 보험에 가입하시기 바랍니다.
※ 합의제도가 있기는 하지만 복잡하고 극히 제한적입니다.
자연재해
ㅇ 홍콩은 자연재해 발생은 없었으나 태풍 시즌(5월~11월)에는 항공기와 페리 운항이 장기간 중단되거나 연착이 됩니다.
유의해야할 지역
ㅇ 홍콩은 비교적 안전한 도시이나 늦은 밤이나 외진 곳에서의 관광은 피하셔야 합니다.
ㅇ 관광객을 대상으로 한 절도사건이 많이 발생하는 곳이므로 항상 소지품에 유의하셔야 합니다.
ㅇ 교통사고의 경우, 합의 전까지는 피해자가 병원비 

ID: sn_07265
metadata: {'chunk_id': 'sn_07265', 'source': 'safety_notice', '작성일': '2026-07-24', '대륙명': '아주', '영문국가명': 'Hongkong', '국가명': '홍콩', '공지유형': '단기', 'ISO코드': 'HK'}
내용: 제목: 태풍 노을(NOUL) 북상에 따른 홍콩 마카오 방문객 안전 유의 공지

ID: sn_07266
metadata: {'공지유형': '단기', '국가명': '홍콩', '대륙명': '아주', '영문국가명': 'Hongkong', 'chunk_id': 'sn_07266', 'ISO코드': 'H